## Аналіз A/B тесту

Маємо проаналізувати дані A/B тесту в популярній [грі Cookie Cats](https://www.facebook.com/cookiecatsgame). Це класична гра-головоломка в стилі «з’єднай три», де гравець повинен з’єднати плитки одного кольору, щоб очистити дошку та виграти рівень. На дошці також зображені співаючі котики :)

Під час проходження гри гравці стикаються з воротами, які змушують їх чекати деякий час, перш ніж вони зможуть прогресувати або зробити покупку в додатку. У цьому блоці завдань ми проаналізуємо результати A/B тесту, коли перші ворота в Cookie Cats було переміщено з рівня 30 на рівень 40. Зокрема, ми проаналізуємо вплив на утримання (retention) гравців. Тобто хочемо зрозуміти чи переміщення воріт на 10 рівнів пізніше якимось чином вплинуло на те, що користувачі перестають грати в гру раніше чи пізніше з точки зору кількості їх днів з моменту встановлення гри.

Будемо працювати з даними з файлу `cookie_cats.csv`. Змінні в даних наступні:

- userid - унікальний номер, який ідентифікує кожного гравця.
- version - чи потрапив гравець в контрольну групу (gate_30 - ворота на 30 рівні) чи тестову групу (gate_40 - ворота на 40 рівні).
- sum_gamerounds - кількість ігрових раундів, зіграних гравцем протягом першого тижня після встановлення
- retention_1 - чи через 1 день після встановлення гравець повернувся і почав грати?
- retention_7 - чи через 7 днів після встановлення гравець повернувся і почав грати?

Коли гравець встановлював гру, його випадковим чином призначали до групи gate_30 або gate_40.

1. Зчитайте дані АВ тесту у змінну `df` та виведіть середнє значення показника показник `retention_7` (утримання на 7 день) по версіям гри. Сформулюйте гіпотезу: яка версія дає краще утримання через 7 днів після встановлення гри?

In [1]:
import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.api as sms
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from math import ceil

%matplotlib inline

In [2]:
df = pd.read_csv('../data/data_statistics/cookie_cats.csv')
df

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True
...,...,...,...,...,...
90184,9999441,gate_40,97,True,False
90185,9999479,gate_40,30,False,False
90186,9999710,gate_30,28,True,False
90187,9999768,gate_40,51,True,False


In [13]:
gate30_reten7_mean = df[df['version'] == 'gate_30']['retention_7'].mean()
gate40_reten7_mean = df[df['version'] == 'gate_40']['retention_7'].mean()


print(f'Mean of gate_30 is {gate30_reten7_mean:.6f} and mean of gate_40 is {gate40_reten7_mean:.6f}')


if gate30_reten7_mean > gate40_reten7_mean:
    print("Gate_30 has a higher mean retention_7 than Gate_40. Provide evidence to support claim that Gate_30 is more effective.")
elif gate30_reten7_mean < gate40_reten7_mean:
    print("Gate_40 has a higher mean retention_7 than Gate_30. Provide evidence to support claim that Gate_40 is more effective.")

Mean of gate_30 is 0.190201 and mean of gate_40 is 0.182000
Gate_30 has a higher mean retention_7 than Gate_40. Provide evidence to support claim that Gate_30 is more effective.


In [7]:
df.groupby('version')['retention_7'].agg(['mean'])

,mean
version,
gate_30,0.190201
gate_40,0.182000


2. Перевірте з допомогою z-тесту аналогічно до прикладу в лекції, чи дає якась з версій гри кращий показник `retention_7` на рівні значущості 0.05. Обчисліть також довірчі інтервали для двох вибірок. Виведіть результат у форматі:
```
z statistic: ...
p-value: ...
Довірчий інтервал 95% для групи control: [..., ...]
Довірчий інтервал 95% для групи treatment: [..., ...]
```
де замість `...` - обчислені значення. В якості висновка дайте відповідь на два питання:  
    1. чи є статистична значущою різниця між поведінкою користувачів у різних версіях гри?   
    2. чи перетинаються довірчі інтервали утримання користувачів з різних версій гри? Про що це каже?  
    
Зверніть увагу, в такому і схожому завданнях ми використовуєм `proportion` Z-тест. Це тому що в нас залежна змінна має бінарне значення (повернеться аби ні користувач, чи клікне або ні користувач в інших ситуаціях - всього два можливих значення в змінної: 0/1, True/False ). Якщо б ми вимірювали скажімо чи є стат. значущою різниця між вагою чоловіків і жінок в певній вибірці, ми б використовувавли функцію `statsmodels.stats.ztest`, бо залежна змінна `вага` є неперервною (тип float, замість типу int чи bool і тільки двох можливих значень).

In [18]:
df.describe()

,userid,sum_gamerounds
count,9.018900e+04,90189.000000
mean,4.998412e+06,51.872457
std,2.883286e+06,195.050858
min,1.160000e+02,0.000000
25%,2.512230e+06,5.000000
50%,4.995815e+06,16.000000
75%,7.496452e+06,51.000000
max,9.999861e+06,49854.000000


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   userid          90189 non-null  int64 
 1   version         90189 non-null  object
 2   sum_gamerounds  90189 non-null  int64 
 3   retention_1     90189 non-null  bool  
 4   retention_7     90189 non-null  bool  
dtypes: bool(2), int64(2), object(1)
memory usage: 2.2+ MB


In [17]:
success = [df[df['version'] == 'gate_30']['retention_7'].sum(), df[df['version'] == 'gate_40']['retention_7'].sum()]
size = [df[df['version'] == 'gate_30']['retention_7'].count(), df[df['version'] == 'gate_40']['retention_7'].count()]

In [22]:
z_statistic, p_value = sms.proportions_ztest(success, size)

print(f"Z-statistic: {z_statistic:.4f}")
print(f"P-value: {p_value:.4f}")


Z-statistic: 3.1644
P-value: 0.0016


In [21]:
from statsmodels.stats.proportion import proportion_confint

ci_gate30 = proportion_confint(success[0], size[0], alpha=0.05, method='normal')
ci_gate40 = proportion_confint(success[1], size[1], alpha=0.05, method='normal')

print(f"95% confidence interval for gate_30: {ci_gate30}")
print(f"95% confidence interval for gate_40: {ci_gate40}")

95% confidence interval for gate_30: (0.18656311652199903, 0.19383956804175934)
95% confidence interval for gate_40: (0.17845430073314686, 0.18554578720019968)


In [24]:
if p_value < 0.05:
    print("Reject the null hypothesis: There is a statistically significant difference between gate_30 and gate_40.")
else:
    print("Fail to reject the null hypothesis: There is no statistically significant difference between gate_30 and gate_40.")

if ci_gate30[1] < ci_gate40[0] or ci_gate30[0] > ci_gate40[1]:
    print("The confidence intervals do not overlap, suggesting a significant difference between gate_30 and gate_40. Provide the evidence that users in gate_30 have higher retention_7 than users in gate_40.")
else:
    print("The confidence intervals overlap, suggesting no significant difference between gate_30 and gate_40. Provide the evidence that users in gate_30 have higher retention_7 than users in gate_40.")


Reject the null hypothesis: There is a statistically significant difference between gate_30 and gate_40.
The confidence intervals do not overlap, suggesting a significant difference between gate_30 and gate_40. Provide the evidence that users in gate_30 have higher retention_7 than users in gate_40.


3. Є ще один тип тестів, який використовується для бінарної метрики як от "зробить юзер дію, чи ні" - тест [**Хі-квадрат**](https://www.bmj.com/about-bmj/resources-readers/publications/statistics-square-one/8-chi-squared-tests) (ще одне [пояснення](https://www.scribbr.com/statistics/chi-square-tests/) тесту з прикладами). В нього інші гіпотези Н0 і Н1 на відміну від z- та t-тестів. А також цей тест можна використовувати, якщо в нас більше за 2 досліджувані групи, тобто в нас не А/В тест, а А/B/C/D, наприклад.  

В **z- та t-тестах** (які відрізняються тим, що ми в першому не знаємо дисперсію генеральної сукупності, але якщо в нас великий набір даних, то ці два тести дають дуже схожі результати) **ми перевіряємо, чи є різниця у середніх показниках по групам користувачів**.  

А в **тесті Хі-квадрат ми перевіряємо чи є звʼязок між групою користувача і тим, чи він зробить цікаву нам дію**. Це ніби дослідження одного і того самого, але дещо різними способами. Для перевірки, можна виконувати кілька тестів (особливо, якщо один дає якийсь непереконливий результат типу р-значення 0.07 - наче і fail to regect H0 на рівні стат значущості 5%, але цікаво, що скажуть інші тести), тож, зробимо і ми тест хі-квадрат та порівняємо його результат з z-тестом.

Про різницю між тестами можна почитати ще [тут](https://stats.stackexchange.com/a/178860) - це просто пояснення користувача стековерфлоу, але там розумні люди сидять.

Для проведення хі-квадрат тесту скористаємось функцією з `scipy.stats` `chi2_contingency` для обчислення статистики хі-квадрат і р-значення для перевірки конкретної гіпотези. У цю функцію вам треба передати таблицю 2х2: кількість випадків для кожної версії гри і значення `retention_7`.

**Задача**: виконайте тест хі-квадрат на рівні значущості 5% аби визначити, чи є залежність між версією гри та тим, чи зайде гравець на 7ий день після встановлення гри.
Тут гіпотези наступні
- Н0: значення retention_7 не залежить від версії гри
- Н1: є залежність між версією гри і значенням retention_7

Виведіть p-значення та зробіть висновок.


In [26]:
df.head(5)

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


In [27]:
table = pd.crosstab(df['version'], df['retention_7'])
print(table)

retention_7  False  True 
version                  
gate_30      36198   8502
gate_40      37210   8279


In [29]:
from scipy.stats import chi2_contingency

chi2, p_value, dof, expected = chi2_contingency(table)

print(f"Chi-squared statistic: {chi2:.4f}")
print(f"P-value: {p_value:.4f}")


if p_value < 0.05:
    print("Reject the null hypothesis: There is a statistically significant association between version and retention_7.")
else:
    print("Fail to reject the null hypothesis: There is no statistically significant association between version and retention_7.")


Chi-squared statistic: 9.9591
P-value: 0.0016
Reject the null hypothesis: There is a statistically significant association between version and retention_7.
